In [ ]:
import csv


# 1️⃣ Charger le mapping
mapping = {}

with open("mapping.csv", newline="", encoding="utf-8") as map_file:
    reader = csv.reader(map_file)
    next(reader)  # retirer si pas d'en-tête
    for old, new in reader:
        mapping[old] = new


# 2️⃣ Lire le fichier comme TEXTE
with open(r"C:\Users\L14\Downloads\ABH_CSV\ALL_TABLES_EXPORT bossoh.csv", "r", encoding="utf-8") as infile:
    content = infile.read()


# 3️⃣ Remplacer toutes les anciennes valeurs
for old, new in mapping.items():
    content = content.replace(old, new)


# 4️⃣ Écrire le nouveau fichier
with open(r"C:\Users\L14\Downloads\ABH_CSV\ALL_TABLES_EXPORT bossoh_copie.csv", "w", encoding="utf-8") as outfile:
    outfile.write(content)

In [3]:
import pandas as pd
from pathlib import Path
from unidecode import unidecode
from datetime import datetime

date = datetime.now()
format_date = "%Y_%m_%d_%H_%M"
date = date.strftime(format_date)
folder = Path(r"C:\Users\L14\Downloads\ABH_CSV\ALL_CSV\VOISINS CTB SIGNES")
MAIN_PATH = Path(r"C:\Users\L14\Downloads\ABH_CSV")
pub_ouv_path = Path(fr"{MAIN_PATH}\PUB OUVERTE_total_fusionne.xlsx")
path_voisins_signes = Path(fr"{MAIN_PATH}\VOISINS_PRESENCE_total_fusionne.xlsx")

files = folder.glob('*.xlsx')

df_pub_ouv = pd.read_excel(pub_ouv_path, engine='openpyxl')
df_voisins_signes = pd.read_excel(path_voisins_signes, engine='openpyxl')
lists = []
for file in files:
    df = pd.read_excel(file, engine='openpyxl')
    
    #print(df.columns)
    if "A ajouter ou Corriger sur DIGIFOR" in df.columns:
        col = df["A ajouter ou Corriger sur DIGIFOR"].astype(str)

        df["indice"] = col.str.split(":").str[0].str.strip()
        df["nom_prenoms"] = col.str.split(":").str[-1].str.strip()
        df["a_signer"] = col.str.contains(r"\(", regex=True, na=False).map({True: "OUI", False: "NON"})
        df["voisin_pers"] = col.str.contains(r"riv\d+", regex=True, na=False).map({True: "OUI", False: "NON"})
        df["cas"] = col.str.contains(r"\(", regex=True, na=False).map({True: "dem", False: ""})
        df["cas_vil"] = col.str.contains(r"\(", regex=True, na=False).map({True: "dem", False: ""})
        df["cas_db"] = col.str.contains(r"\(", regex=True, na=False).map({True: "dem", False: ""})
        mask_df = (df["a_signer"] == "NON") & (df["voisin_pers"] == "OUI")
        #df = df.loc[mask_df, :]

    lists.append(df)
    #df.to_excel(f"{folder}/REM_SIGNATURE_CTB/{file.stem}_pub_ouv_{date}.xlsx")

df_result = pd.concat(lists)
df_result.to_excel(fr"C:\Users\L14\Downloads\ABH_CSV\ALL_CSV\VOISINS CTB SIGNES\REM_SIGNATURE_CTB_total_fusionne.xlsx")


In [4]:
from unidecode import unidecode
from rapidfuzz import fuzz
import pandas as pd
folder = Path(r"C:\Users\L14\Downloads\ABH_CSV\ALL_CSV\VOISINS CTB SIGNES")
MAIN_PATH = Path(r"C:\Users\L14\Downloads\ABH_CSV")
pub_ouv_path = Path(fr"{MAIN_PATH}\PUB OUVERTE_total_fusionne.xlsx")
path_voisins_signes = Path(fr"{MAIN_PATH}\VOISINS_PRESENCE_total_fusionne.xlsx")

files = folder.glob('*.xlsx')

df_pub_ouv = pd.read_excel(pub_ouv_path, engine='openpyxl')
df_voisins_signes = pd.read_excel(path_voisins_signes, engine='openpyxl')

def clean_name(x):
    if pd.isna(x):
        return ""
    return unidecode(str(x)).lower().strip()



df_voisins_ctb_signes = pd.read_excel(fr"C:\Users\L14\Downloads\ABH_CSV\ALL_CSV\VOISINS CTB SIGNES\REM_SIGNATURE_CTB_total_fusionne.xlsx", engine='openpyxl')
df_pub_ouv = pd.read_excel(fr"C:\Users\L14\Downloads\Liste demande en PUB.xlsx", engine='openpyxl')

df_voisins_ctb_signes = pd.merge(df_voisins_ctb_signes,df_pub_ouv,left_on='code',right_on="num_demand",how='inner') 

df_voisins_ctb_signes["nom_prenoms"] = df_voisins_ctb_signes["nom_prenoms"].apply(clean_name)
df_voisins_ctb_signes["nom_prenoms"] = df_voisins_ctb_signes["nom_prenoms"].astype(str).str.split("(",n=1).str[0]
df_voisins_signes["nameOfPerson"] = df_voisins_signes["nameOfPerson"].apply(clean_name)

df_voisins_ctb_signes.loc[:, "village"] = (df_voisins_ctb_signes["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )
df_voisins_signes.loc[:, "village"] = (df_voisins_signes["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )
df_voisins_ctb_signes_grouped = {
    k: v for k, v in df_voisins_ctb_signes.groupby("village")
}
df_voisins_signes_grouped = {
    k: v for k, v in df_voisins_signes.groupby("village")
}

results = []
results_ctb_sans_match = []

villages = set(df_voisins_ctb_signes_grouped.keys()) & set(df_voisins_signes_grouped.keys())

matched_ctb_indices = set()
matched_sig_indices = set()

for vil in villages:

    df_ctb_village = df_voisins_ctb_signes_grouped[vil]
    df_signes_village = df_voisins_signes_grouped[vil]
    #print(df_signes_village.columns)
    used_indices = set()

    for idx_ctb, row_ctb in df_ctb_village.iterrows():

        nom_ctb = str(row_ctb["nom_prenoms"])
        requestDate = str(row_ctb["requestDate"])
        code = str(row_ctb["code"])
        indice = str(row_ctb["indice"])
        

        best_score = 0
        best_idx = None
        is_perfect_match = False
        is_representant_match = False

        for idx_sig, row_sig in df_signes_village.iterrows():

            #if idx_sig in used_indices:
            #    continue

            nom_sig = str(row_sig["nameOfPerson"])
            nom_repr = str(row_sig["representant_clean"])
            
            score = fuzz.token_set_ratio(nom_ctb, nom_sig)
            score_repr = fuzz.token_set_ratio(nom_ctb, nom_repr)
            if score == 100 or score_repr == 100:
                best_score = 100
                best_idx = idx_sig
                is_representant_match = False if score == 100 else True
                break

            if score > best_score :
                best_score = score
                best_idx = idx_sig
                is_representant_match = False
            if score_repr > best_score:
                best_score = score_repr
                best_idx = idx_sig
                is_representant_match = True

        # classification
        if best_score >= 90:
            match_quality = "match_parfait"
        elif best_score >= 80:
            match_quality = "match_probable"
        elif best_score >= 70:
            match_quality = "a_verifier"
        else:
            match_quality = "pas_de_match"

        if best_idx is not None :
            # and best_score >= 70
            used_indices.add(best_idx)
            matched_ctb_indices.add(idx_ctb)
            matched_sig_indices.add(best_idx)

            results.append({
                "code": code,
                "requestDate": requestDate,
                "indice": indice,
                "village": vil,
                "nom_ctb": nom_ctb,
                "is_representant_match": is_representant_match,
                "nom_sig_original": df_signes_village.loc[best_idx, "nameOfPersonOriginal"],
                "nom_sig": df_signes_village.loc[best_idx, "nameOfPerson"],
                "nom_repr_original": df_signes_village.loc[best_idx, "representantOriginal"],
                "nom_repr": df_signes_village.loc[best_idx, "representant_clean"],
                "code_sig": df_signes_village.loc[best_idx, "codePresence"],
                "number_cni": df_signes_village.loc[best_idx, "numberCNI"],
                "score": best_score,
                "qualite_match": match_quality,
                "signatory_photo": df_signes_village.loc[best_idx, "signatoryPhoto"]
            })

        else:
            # CTB sans match
            results.append({
                "code": code,
                "requestDate": requestDate,
                "indice": indice,
                "village": vil,
                "nom_ctb": nom_ctb,
                "is_representant_match": False,
                "nom_sig_original": None,
                "nom_sig": None,
                "nom_repr": None,
                "nom_repr_original": None,
                "code_sig": None,
                "number_cni": None,
                "score": best_score,
                "qualite_match": "ctb_sans_match",
                "signatory_photo": None
            })

# ajouter les voisins signés sans correspondance
#for idx_sig, row_sig in df_voisins_signes.iterrows():

    #if idx_sig not in matched_sig_indices:

        """results.append({
            "village": row_sig["village"],
            "nom_ctb": None,
            "nom_sig": row_sig["nom_prenoms"],
            "score": None,
            "qualite_match": "signes_sans_match"
        })"""

df_matches = pd.DataFrame(results) 

mask_ctb = (
    df_matches["nom_ctb"].astype(str).str.contains(
        r"foret|village|famille|aucun", case=False, na=False
    )
    | df_matches["indice"].astype(str).str.contains(
        r"par|zone", case=False, na=False
    )
)

df_matches_ = df_matches[~(df_matches["qualite_match"].astype(str).isin(["pas_de_match", "ctb_sans_match"])) ]
df_matches_.to_csv("matches_repr_test.csv", sep=";", encoding="utf-8-sig")
df_matches.loc[~mask_ctb,:].to_csv("extraction_matches.csv", sep=";", encoding="utf-8-sig")
df_matches["qualite_match"].value_counts()
        
        

qualite_match
match_parfait     485
pas_de_match       27
match_probable     27
a_verifier          9
ctb_sans_match      2
Name: count, dtype: int64

In [ ]:
print(sorted({'soumahoro_kpan_henri', 'diomande_sassa', 'mamadou_bamba', 'gondo_diomande', 'diomande_louale', 'tiemoko_diomande', 'soumahoro_wohi'}))

In [ ]:
df_voisins_ctb_signes = pd.read_excel(fr"C:\Users\L14\Downloads\ABH_CSV\ALL_CSV\VOISINS CTB SIGNES\REM_SIGNATURE_CTB_total_fusionne.xlsx", engine='openpyxl')

mask_dem = df_voisins_ctb_signes["nom_prenoms"].str.contains(r"\(", regex=True, na=False)

df_voisins_signes.loc[:, "village"] = (df_voisins_signes["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )

df_voisins_ctb_signes["nom_prenoms"] = (
    df_voisins_ctb_signes["nom_prenoms"]
    .astype(str)
    .str.replace(r"\(.*\)", "", regex=True)
    .str.strip()
)

df_voisins_signes.loc[:, "code_vois"] = (
        df_voisins_signes["village"].astype(str).str.strip()
        + "-"
        + df_voisins_signes["nameOfPerson"].astype(str).str.strip()
)

df_voisins_signes = (
    df_voisins_signes
    .drop_duplicates("code_vois", keep="first")
)

df_voisins_ctb_signes.loc[:, "village"] = (df_voisins_ctb_signes["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )


df_voisins_ctb_signes.loc[:, "code_vois_old"] = (
    df_voisins_ctb_signes["village"].astype(str).str.strip()
    + "-"
    + df_voisins_ctb_signes["nom_prenoms"].astype(str).str.strip()
)

df_voisins_ctb_signes = df_voisins_ctb_signes.merge(
    df_voisins_signes[["code_vois", "numberCNI", "signatoryPhoto","representant"]],
    left_on="code_vois_old",
    right_on="code_vois",
    how="left",
    suffixes=("","_autre")
)

df_voisins_ctb_signes["cas_vil"] = (
        df_voisins_ctb_signes["cas_vil"]
        .replace("", pd.NA)
        .fillna("vil")
)

df_voisins_ctb_signes["cas"] = (
        df_voisins_ctb_signes["cas"]
        .replace("", pd.NA)
        .fillna("vil")
)

df_voisins_ctb_signes_unique = df_voisins_ctb_signes.drop_duplicates(subset="nom_prenoms")
df_voisins_signes_unique = df_voisins_signes.drop_duplicates(subset="nameOfPerson")

print(df_voisins_signes_unique.columns)
print(df_voisins_signes_unique.shape)

df_voisins_ctb_signes = df_voisins_ctb_signes.merge(
    df_voisins_signes_unique[["code_vois", "nameOfPerson", "numberCNI", "signatoryPhoto","representant"]],
    left_on="nom_prenoms",
    right_on="nameOfPerson",
    how="left",
    suffixes=("","_autre")
)

df_voisins_ctb_signes["numberCNI"] = df_voisins_ctb_signes["numberCNI"].fillna(df_voisins_ctb_signes["numberCNI_autre"])
df_voisins_ctb_signes["signatoryPhoto"] = df_voisins_ctb_signes["signatoryPhoto"].fillna(df_voisins_ctb_signes["signatoryPhoto_autre"])
df_voisins_ctb_signes["representant"] = df_voisins_ctb_signes["representant"].fillna(df_voisins_ctb_signes["representant_autre"])

df_voisins_ctb_signes["cas_db"] = (
        df_voisins_ctb_signes["cas_db"]
        .replace("", pd.NA)
        .fillna("db")
)

df_voisins_ctb_signes["cas"] = (
        df_voisins_ctb_signes["cas"]
        .replace("", pd.NA)
        .fillna("db")
)

df_voisins_ctb_signes.drop(columns=["representant_autre", "numberCNI_autre", "signatoryPhoto_autre"],errors="ignore", inplace=True)

df_voisins_ctb_signes.to_csv(fr"C:\Users\L14\Desktop\CONTROLE_FINAL\VOISINS CTB SIGNES_total_fusionne_pub_ouv_signe_{date}.csv", sep=";", encoding="utf-8-sig")


In [ ]:
import pandas as pd
from pathlib import Path
from unidecode import unidecode

def normalize(name):
    return ' '.join(str(name).lower().split()).strip()

def format_name(name):
    def normalize(name):
        return '_'.join(str(name).lower().replace(';'," ").split()).strip()
    
    values = name.split(';',2)
    if len(values) > 1 :
        if normalize(values[0]) == normalize(values[1]):
            return values[0]
        else :
            return " ".join(values[:2])
    else :
        return name.replace(';'," ")
  
etat = pd.read_csv(Path(r"C:\Users\L14\Downloads\ABH_CSV\data_tokpi_19_03_26_clean.csv"))
folder = Path(r"C:\Users\L14\Downloads\ABH_CSV\CONTROL_FINAL\FICHE_PRESENCE")

files = folder.glob('*.csv')

for file in files:
    df = pd.read_csv(file)
    df["nom_prenoms"] = df["nom_prenoms"].apply(format_name)
    etat = etat.merge(df[["code","nom_prenoms"]], on="code", how="left", suffixes=("", "_autre"))
    etat["nom_prenoms"] = etat["nom_prenoms"].fillna(etat["nom_prenoms_autre"])
    etat.drop(columns=["nom_prenoms_autre"], inplace=True)
etat.to_csv(r"C:\Users\L14\Downloads\ABH_CSV\data_tokpi_19_03_26_clean.csv", index=False)


C:\Users\L14\AppData\Local\Temp\ipykernel_6208\2161629376.py:21: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  etat = pd.read_csv(Path(r"C:\Users\L14\Downloads\ABH_CSV\data_tokpi_19_03_26.csv"))


In [12]:
import pandas as pd
from pathlib import Path
from unidecode import unidecode
etat = pd.read_excel(Path(r"C:\Users\L14\Downloads\ABH_CSV\VOISINS_PRESENCE_total_fusionne.xlsx"), engine='openpyxl')
    
STOPWORDS = ["representant", "voisin_limitrophes", "voisin_limitrophe", "voisine", "voisin","voisins",  "_et_", "_de_", "_du_", ",", "/" , "limitrophe", "-"]

def clean_name_sans_match(x):
    if '_' not in x:
         return ""
    for w in STOPWORDS:
        x = x.replace(w, " ")
    words = x.replace(r"_", " ").strip()
    words = words.split()
    words = [w for w in words if w not in STOPWORDS]
    return "_".join(words)

etat["representant"] = etat["representant"].astype(str).apply(clean_name_sans_match)

etat.to_excel(r"C:\Users\L14\Downloads\ABH_CSV\VOISINS_PRESENCE_total_fusionne.xlsx", index=False, engine='openpyxl')